In [ ]:
# -*- coding: utf-8 -*-
"""
Sudden Cardiac Death Prediction using Feature Engineering and Stacked Ensemble Learning.

This script extracts HRV and SPFF features from ECG segments preceding SCD events
in the SDDB database, performs enhanced EDA, trains a stacking ensemble model (including RF,
SVM, XGBoost, and MLP), evaluates its performance using various metrics and threshold
analysis, and saves the results. Data quality checks are added during feature extraction.

Author: [Shrikant Kabade]
Date: 2025-04-27
Version: 1.6 (Fixed NameError by moving utility function definitions)
"""

import os
import numpy as np
import pandas as pd
import wfdb
import joblib
import logging
import random
import subprocess
from datetime import datetime
import json # To save config
import copy # To copy config for saving

# Scipy imports
from scipy.signal import find_peaks, welch
from scipy.integrate import trapezoid
# sklearn imports
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import FastICA # Not used currently, kept from original
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression # Corrected import from sklearn.linear_network
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay, f1_score, recall_score, precision_score, auc, precision_recall_curve # Import precision_recall_curve

# imblearn imports
from imblearn.over_sampling import SMOTE

# plotting
import matplotlib.pyplot as plt
import seaborn as sns

# xgboost
import xgboost as xgb

# --- Configuration ---
CONFIG = {
    "data": {
        "data_dir": '/content/drive/MyDrive/sddb',
        "fs": 250,
        "window_minutes": 10,
        "pos_offsets_min": [1,2,3], # Offsets *before* the event for positive samples
        "neg_offsets_min": [5,10,15], # Offsets *before* the event for negative samples
        "min_segment_len_sec": 60,
        "min_rr_intervals": 10,
        "ecg_channel": 0,
        "event_annotations": ('V','F'), # Ventricular Flutter/Fibrillation events in SDDB often mark SCD
        # Added data quality checks on RR intervals
        "rr_mean_min_ms": 300, # Corresponds to max HR ~200 bpm
        "rr_mean_max_ms": 1500, # Corresponds to min HR ~40 bpm
        "rr_std_max_ms": 500 # Arbitrary threshold for excessive variation, indicates noise/severe artifact
    },
    "preprocessing": {"test_size":0.2, "use_smote":True},
    "modeling": {
        "cv_folds":5, # For stacking internal CV and overall evaluation
        "estimators":{
            "rf": {"n_estimators":150, "class_weight":"balanced", "n_jobs":-1},
            "svm": {"probability":True, "kernel":"rbf", "gamma":"scale", "class_weight":"balanced"},
            "xgb": {"n_estimators":150, "eval_metric":"logloss", "objective":"binary:logistic", "use_label_encoder": False, "n_jobs":-1}, # Added use_label_encoder=False for XGBoost 1.x+
            "mlp": {"hidden_layer_sizes":(64,32), "activation":"relu", "solver":"adam", "alpha":1e-4,
                     "learning_rate_init":1e-3, "max_iter":300, "early_stopping":True, "n_iter_no_change":10}
        },
        "final_estimator": {"solver":"liblinear", "class_weight":"balanced"},
        "stack_method":"predict_proba", # Can also be 'predict'
        "passthrough":False, # Pass original features to meta-learner? False is typical.
        "n_jobs":-1
    },
    "evaluation": {
        # Storing function objects directly here is problematic for JSON saving
        # We will map names to functions when needed
        "metrics_to_report": ["Accuracy", "AUC", "F1-Score", "Recall", "Precision"], # Names of metrics to calculate and report
        "plot_cm": True,
        "plot_feat_imp": True,
        "plot_roc_pr": True, # Added ROC and PR curve plotting
        "feat_imp_top_n": 20,
        "optimal_threshold_metric": "f1" # Metric to optimize when finding threshold ('f1', 'recall', 'precision', 'accuracy')
        },
    "experiment": {
        "random_state":42,
        "log_level":logging.INFO,
        "run_eda":True,
        "save_results":True,
        "results_dir":"./scd_results",
        "run_ts":datetime.now().strftime('%Y%m%d_%H%M%S')
    }
}

# Map metric names to actual functions for evaluation
METRIC_FUNCTIONS = {
    "Accuracy": accuracy_score,
    "AUC": roc_auc_score,
    "F1-Score": f1_score,
    "Recall": recall_score,
    "Precision": precision_score
}


# Logging
logging.basicConfig(level=CONFIG['experiment']['log_level'], format='%(asctime)s - %(levelname)s - %(message)s')
random.seed(CONFIG['experiment']['random_state'])
np.random.seed(CONFIG['experiment']['random_state'])


# --- Utility functions ---
def ensure_dir(path):
    """Ensures a directory exists."""
    if not os.path.isdir(path):
        os.makedirs(path, exist_ok=True)

def download_data():
    """Downloads the SDDB database if not already present."""
    d = CONFIG['data']['data_dir']
    db = 'sddb'
    if not os.path.isdir(d) or not os.listdir(d):
        logging.info(f"Data directory {d} not found or empty. Attempting download.")
        try:
            # Check if wfdb and physionet-client are installed
            subprocess.run(['pip', 'show', 'wfdb'], check=True, capture_output=True)
            subprocess.run(['pip', 'show', 'physionet-client'], check=True, capture_output=True)
        except subprocess.CalledProcessError:
            logging.info("Installing wfdb and physionet-client...")
            subprocess.run(['pip','install','wfdb','physionet-client','-q'], check=True)

        try:
            subprocess.run(['physionet-client','download',db,'--output',d], check=True)
            logging.info(f"Downloaded {db} to {d}")
        except subprocess.CalledProcessError as e:
            logging.error(f"Failed to download data: {e}")
            raise SystemExit("Data download failed. Please check your network connection and PhysioNet client installation.")
    else:
        logging.info(f"Data directory {d} exists and is not empty.")

# --- End Utility functions ---


# Set up results directory
RESULTS_PATH = os.path.join(CONFIG['experiment']['results_dir'], CONFIG['experiment']['run_ts'])
if CONFIG['experiment']['save_results']:
    ensure_dir(RESULTS_PATH) # This call will now find the definition above
    logging.info(f"Results will be saved to: {RESULTS_PATH}")


# Define feature names (MUST match the order and number of features generated by spff and hrv)
# SPFF features: 4 stats (mean, med, min, max) for each segment.
# Assuming max 1 segment of size n, max 2 of n/2, max 4 of n/4 = 1+2+4=7 segments max * 4 stats = 28
SPFF_FEAT_NAMES = [f'SPFF_{i}' for i in range(28)] # Approximation based on ideal case
HRV_FEAT_NAMES = ['mean_rr', 'std_rr', 'rms_diff_rr', 'pnn50', 'lf_power', 'hf_power', 'lf_hf_ratio']

# Initial placeholder. This will be updated after feature extraction
FEATURE_NAMES = SPFF_FEAT_NAMES + HRV_FEAT_NAMES


# Global counter for segments skipped due to non-finite values
segments_skipped_non_finite = 0

# Feature extraction
def extract_rr(seg, fs):
    """Extracts RR intervals from an ECG segment."""
    # Simple R-peak detection - note this can be a source of errors
    # More robust methods (e.g., Pan-Tompkins) would be better but require more code
    global segments_skipped_non_finite
    try:
        # Ensure segment is numpy array and finite
        seg = np.asarray(seg)
        if not np.isfinite(seg).all(): # Check for NaN/inf
             segments_skipped_non_finite += 1 # Increment counter
             # Removed the warning here to declutter logs
             return np.array([]) # Skip segment

        # Simple peak finding. Adjust properties like height or prominence if needed for better robustness.
        peaks, _ = find_peaks(seg, distance=int(fs * 0.6)) # Minimum distance ~100bpm
        rr_intervals = np.diff(peaks) / fs # RR intervals in seconds
        return rr_intervals
    except Exception as e:
        logging.error(f"Error during R-peak detection or RR extraction: {e}")
        return np.array([])

def is_valid_rr_segment(rr_intervals, config):
    """Checks if the extracted RR intervals meet basic quality criteria."""
    if len(rr_intervals) < config['min_rr_intervals']:
        return False, "Too few RR intervals"

    rr_ms = rr_intervals * 1000 # Convert to milliseconds for checks

    if len(rr_ms) < 2: # Need at least two RR intervals to calculate std dev etc.
         return False, "Less than 2 RR intervals for validation"

    # Ensure RR intervals are positive and finite before calculating stats
    if not (rr_ms > 0).all() or not np.isfinite(rr_ms).all():
         return False, "RR intervals contain non-positive or non-finite values after diff (unexpected)"

    mean_rr = np.mean(rr_ms)
    std_rr = np.std(rr_ms)

    if mean_rr < config['rr_mean_min_ms'] or mean_rr > config['rr_mean_max_ms']:
        return False, f"Mean RR ({mean_rr:.1f}ms) outside physiological range"
    if std_rr > config['rr_std_max_ms']:
         return False, f"Std Dev RR ({std_rr:.1f}ms) indicates excessive variation"

    # Add other checks if needed, e.g., percentage of outliers or artifacts

    return True, "" # Segment is considered valid

SPFF_FUNCS = [np.mean, np.median, np.min, np.max]
def spff(rr):
    """Calculates Short-term Periodical Fluctuation Features (SPFF)."""
    feats = []
    n = len(rr)

    # Define segment lengths and strides. Ensure segment_len is at least 1.
    # Calculate segment lengths based on the current segment size n
    segments_info = [
        (n, n),      # Window size n, stride n (1 segment)
        (n // 2, n // 2), # Window size n//2, stride n//2 (up to 2 segments)
        (n // 4, n // 4)  # Window size n//4, stride n//4 (up to 4 segments)
    ]

    for seg_len, stride in segments_info:
        if seg_len < 1: continue # Skip if segment length is zero or negative
        # Ensure stride is at least 1 if seg_len is meaningful
        stride = max(1, stride) if seg_len > 0 else stride # Ensure stride is not zero for meaningful lengths

        for i in range(0, n - seg_len + 1, stride):
            segment = rr[i : i + seg_len]
            if len(segment) > 0: # Ensure segment is not empty
                for func in SPFF_FUNCS:
                    # Handle potential errors in stats calculation if segment is tiny or has NaNs
                    try:
                        feats.append(func(segment))
                    except Exception:
                         feats.append(np.nan) # Append NaN if calculation fails

    # Pad feature vector to consistent size if needed
    max_spff_feats = 28 # Based on ideal case n being large and multiple of 4
    if len(feats) < max_spff_feats:
        feats.extend([np.nan] * (max_spff_feats - len(feats)))
    # If somehow more are created (unlikely with this logic), truncate
    feats = feats[:max_spff_feats]

    return feats


def hrv(rr, fs_approx=4.0): # Increased fs_approx, common for interpolated RR
    """Calculates Heart Rate Variability (HRV) features."""
    feats = []
    # Ensure rr is not empty or too short for calculations
    if len(rr) < 2:
        return [np.nan] * 7 # Return NaNs if not enough data for basic stats

    # time domain
    try:
        diff_rr = np.abs(np.diff(rr))
        feats.append(np.mean(rr)) # mean_rr (seconds)
        feats.append(np.std(rr)) # std_rr (seconds)
        # rmssd requires at least one difference
        feats.append(np.sqrt(np.mean(diff_rr**2)) if len(diff_rr) > 0 else np.nan) # rmssd (seconds)

        # pNN50 - Percentage of successive RR intervals that differ by more than 50ms (0.05s)
        pnn50_count = np.sum(diff_rr > 0.05)
        feats.append(pnn50_count / len(diff_rr) if len(diff_rr) > 0 else 0)

    except Exception as e:
        logging.warning(f"Error during HRV time domain calculation: {e}. Returning NaNs for time features.")
        feats.extend([np.nan] * 4) # Add NaNs for mean, std, rmssd, pnn50

    # freq domain (Welch on RR intervals treated as sampled at fs_approx)
    # This is an approximation. For true frequency domain, RR series should ideally be
    # interpolated or methods like Lomb-Scargle used.
    # Need at least 2 RR intervals for time domain, more for meaningful frequency domain
    if len(rr) > 10: # Arbitrary threshold for meaningful frequency analysis
        try:
            # Ensure nperseg and noverlap are valid for the length of rr
            # nperseg must be less than or equal to len(rr)
            nperseg = min(256, len(rr) // 2 * 2)
            if nperseg < 4 and len(rr) >= 4: # Ensure nperseg is at least 4 if possible for basic frequency analysis
                 nperseg = (len(rr) // 2 * 2) if (len(rr) // 2 * 2) >= 4 else (len(rr) // 1 * 2) if (len(rr) // 1 * 2) >= 4 else len(rr) # Try to get at least 4

            noverlap = min(128, nperseg // 2) if nperseg > 1 else 0


            if nperseg < 4: # Cannot compute Welch meaningfully with nperseg < 4
                 logging.debug(f"RR length {len(rr)} too short for Welch (nperseg={nperseg}). Returning NaNs for freq features.")
                 feats.extend([np.nan] * 3)
            else:
                f, p = welch(rr, fs=fs_approx, nperseg=nperseg, noverlap=noverlap)
                # Define frequency bands
                # vlf_band = (0.003, 0.04) # Often not reliable for short segments
                lf_band = (0.04, 0.15)
                hf_band = (0.15, 0.4)

                # Integrate power in bands
                # Ensure frequency points exist within the band
                lf_mask = (f >= lf_band[0]) & (f < lf_band[1])
                hf_mask = (f >= hf_band[0]) & (f < hf_band[1])

                # Check if bands have at least 2 points for trapezoidal integration
                lf_power = trapezoid(p[lf_mask], f[lf_mask]) if np.sum(lf_mask) > 1 else 0
                hf_power = trapezoid(p[hf_mask], f[hf_mask]) if np.sum(hf_mask) > 1 else 0

                feats.append(lf_power)
                feats.append(hf_power)
                feats.append(lf_power / (hf_power + 1e-9)) # Avoid division by zero

        except Exception as e:
            logging.warning(f"Error during HRV frequency domain calculation: {e}. Returning NaNs for freq features.")
            feats.extend([np.nan] * 3) # Add NaNs for lf, hf, lf/hf
    else:
        logging.debug(f"RR length {len(rr)} too short for meaningful frequency analysis. Returning NaNs for freq features.")
        feats.extend([np.nan] * 3)


    # Ensure exactly 7 HRV features are returned (pad with NaN if needed due to errors)
    while len(feats) < 7:
        feats.append(np.nan)
    return feats[:7]


# Load and extract
def extract_all_features():
    """Extracts features from all relevant segments in the database."""
    download_data()
    X, y = [], []
    cfg = CONFIG['data']
    processed_count = 0
    skipped_count_validity = 0 # Skipped due to validity checks (min_rr, mean/std range)
    total_segments_considered = 0
    # The global counter segments_skipped_non_finite tracks segments skipped due to NaN/inf

    # Reset global counter for this run
    global segments_skipped_non_finite
    segments_skipped_non_finite = 0


    # Collect all record names first
    record_names = [fname[:-4] for fname in os.listdir(cfg['data_dir']) if fname.endswith('.dat')]
    logging.info(f"Found {len(record_names)} potential records.")

    for base in record_names:
        path = os.path.join(cfg['data_dir'], base)
        # Check for .atr annotation file
        if not os.path.isfile(path + '.atr'):
            logging.debug(f"Skipping record {base}: .atr file not found.")
            continue

        try:
            rec = wfdb.rdrecord(path)
            ann = wfdb.rdann(path, 'atr')
        except Exception as e:
            logging.error(f"Error reading record or annotation {base}: {e}")
            continue

        # Check if the required ECG channel exists
        if cfg['ecg_channel'] >= rec.n_sig:
             logging.warning(f"Record {base} does not have channel {cfg['ecg_channel']}. Skipping.")
             continue

        sig = rec.p_signal[:, cfg['ecg_channel']]
        fs = rec.fs # Use actual sampling frequency

        # Find event annotations ('V', 'F')
        event_samples = sorted([s for s, sym in zip(ann.sample, ann.symbol) if sym in cfg['event_annotations']])

        # Process only the last few potential events as specified in the original script
        relevant_events = event_samples[-3:] if event_samples else []

        if not relevant_events:
            logging.debug(f"No target events ('V' or 'F') found in record {base}.")
            continue

        for ev_sample in relevant_events:
            # Extract positive samples (segments ending shortly before the event)
            for m_offset in cfg['pos_offsets_min']:
                end_sample = ev_sample - m_offset * 60 * fs
                start_sample = max(0, end_sample - cfg['window_minutes'] * 60 * fs)

                if end_sample <= start_sample: # Segment too short or invalid range
                     logging.debug(f"Skipping positive segment for {base} at offset {m_offset} min: invalid time range.")
                     continue

                segment = sig[start_sample : end_sample]
                total_segments_considered += 1

                if len(segment) >= cfg['min_segment_len_sec'] * fs:
                    rr_intervals = extract_rr(segment, fs) # extract_rr increments segments_skipped_non_finite if needed
                    # If extract_rr returned empty because of non-finite values, is_valid will be False
                    is_valid, reason = is_valid_rr_segment(rr_intervals, cfg)
                    if is_valid:
                        features = spff(rr_intervals) + hrv(rr_intervals, fs_approx=4.0)
                        # Features can still contain NaNs if functions failed for complex reasons not caught by is_valid
                        X.append(features)
                        y.append(1) # Positive label
                        processed_count += 1
                    else:
                        # This segment was skipped because of issues detected by is_valid_rr_segment (excluding non-finite handled in extract_rr)
                        logging.debug(f"Skipping positive segment for {base} at offset {m_offset} min: {reason}")
                        skipped_count_validity += 1
                else:
                    logging.debug(f"Skipping positive segment for {base} at offset {m_offset} min: too short ({len(segment)} samples).")
                    skipped_count_validity += 1 # Also count short segments here

            # Extract negative samples (segments further before the event)
            for m_offset in cfg['neg_offsets_min']:
                 # Using the original code's interpretation for negative segments
                 neg_end_sample = ev_sample - cfg['window_minutes']*60*fs - m_offset*60*fs
                 neg_start_sample = max(0, neg_end_sample - cfg['window_minutes']*60*fs)

                 if neg_end_sample <= neg_start_sample or neg_end_sample <= 0:
                      logging.debug(f"Skipping negative segment for {base} at offset {m_offset} min: invalid time range or before record start.")
                      continue

                 segment = sig[neg_start_sample : neg_end_sample]
                 total_segments_considered += 1

                 if len(segment) >= cfg['min_segment_len_sec'] * fs:
                     rr_intervals = extract_rr(segment, fs) # extract_rr increments segments_skipped_non_finite if needed
                     # If extract_rr returned empty because of non-finite values, is_valid will be False
                     is_valid, reason = is_valid_rr_segment(rr_intervals, cfg)
                     if is_valid:
                         features = spff(rr_intervals) + hrv(rr_intervals, fs_approx=4.0)
                         # Features can still contain NaNs if functions failed for complex reasons not caught by is_valid
                         X.append(features)
                         y.append(0) # Negative label
                         processed_count += 1
                     else:
                         # This segment was skipped because of issues detected by is_valid_rr_segment (excluding non-finite handled in extract_rr)
                         logging.debug(f"Skipping negative segment for {base} at offset {m_offset} min: {reason}")
                         skipped_count_validity += 1
                 else:
                    logging.debug(f"Skipping negative segment for {base} at offset {m_offset} min: too short ({len(segment)} samples).")
                    skipped_count_validity += 1 # Also count short segments here

        logging.debug(f"Finished processing record {base}.") # Changed to debug to reduce log spam

    logging.info(f"Feature extraction complete.")
    logging.info(f"Total segments considered: {total_segments_considered}")
    logging.info(f"Segments skipped due to non-finite values in signal: {segments_skipped_non_finite}")
    logging.info(f"Segments skipped due to RR interval validity/length checks: {skipped_count_validity}")
    logging.info(f"Successfully processed segments: {processed_count}")


    X_arr = np.array(X)
    y_arr = np.array(y)

    # Check if the number of features is consistent after extraction
    if X_arr.shape[0] > 0:
        expected_num_features_approx = len(SPFF_FEAT_NAMES) + len(HRV_FEAT_NAMES) # Approximate
        actual_num_features = X_arr.shape[1]

        if actual_num_features != expected_num_features_approx:
            logging.warning(f"Feature dimension mismatch! Expected approx {expected_num_features_approx}, got {actual_num_features}.")
            # Update FEATURE_NAMES based on the actual number of features extracted
            logging.warning(f"Updating FEATURE_NAMES to generic names based on actual count ({actual_num_features}).")
            global FEATURE_NAMES # Need to modify the global variable
            FEATURE_NAMES = [f'Feature_{i}' for i in range(actual_num_features)]
        # Else: actual_num_features matches approx, keep original FEATURE_NAMES

    else:
        logging.error("No features extracted. Cannot proceed.")
        raise SystemExit("No valid data segments found or features extracted.")

    return X_arr, y_arr, FEATURE_NAMES # Return actual feature names

# EDA
def run_eda(X_orig_train, y_orig_train, feature_names):
    """Performs Exploratory Data Analysis on the original training data."""
    logging.info("Running EDA on original training data...")
    df = pd.DataFrame(X_orig_train, columns=feature_names)
    df['label'] = y_orig_train

    # Class distribution
    plt.figure(figsize=(6, 4))
    sns.countplot(x='label', data=df)
    plt.title('Class Distribution (Original Training Set)')
    plt.xlabel('Label (0: Non-SCD, 1: SCD)')
    plt.ylabel('Count')
    if CONFIG['experiment']['save_results']:
        plt.savefig(os.path.join(RESULTS_PATH, 'eda_class_distribution_train_orig.png'))
    plt.show()

    # Feature distributions (histograms or box plots)
    # Select a subset of features for visualization, otherwise too many plots
    num_features_to_plot = min(10, len(feature_names))
    # Select features - try to include some from SPFF and HRV if possible, otherwise random
    selected_features = []
    # Try to select features by their original names if they exist
    spff_candidates = [f for f in FEATURE_NAMES if f.startswith('SPFF_')]
    hrv_candidates = [f for f in FEATURE_NAMES if f in HRV_FEAT_NAMES] # Use the original HRV names

    # Prioritize known names if they are present in the actual features
    available_spff = [f for f in spff_candidates if f in df.columns]
    available_hrv = [f for f in hrv_candidates if f in df.columns]
    available_other = [f for f in FEATURE_NAMES if f not in spff_candidates and f not in hrv_candidates and f in df.columns]


    selected_features.extend(available_spff[:min(num_features_to_plot//3, len(available_spff))])
    selected_features.extend(available_hrv[:min(num_features_to_plot//3, len(available_hrv))])

    # Fill the rest with random features from the remaining available ones
    remaining_features = [f for f in available_other if f not in selected_features]
    if len(selected_features) < num_features_to_plot and len(remaining_features) > 0:
         num_to_add = min(num_features_to_plot - len(selected_features), len(remaining_features))
         selected_features.extend(random.sample(remaining_features, num_to_add))

    # Fallback: if still not enough features selected (e.g., less than num_features_to_plot features available)
    if len(selected_features) < num_features_to_plot and len(df.columns) > 1: # >1 to exclude 'label'
        already_selected_in_df = [f for f in selected_features if f in df.columns]
        all_df_features = [f for f in df.columns if f != 'label']
        remaining_df_features = [f for f in all_df_features if f not in already_selected_in_df]
        num_to_add = min(num_features_to_plot - len(selected_features), len(remaining_df_features))
        if num_to_add > 0:
             selected_features.extend(random.sample(remaining_df_features, num_to_add))


    if not selected_features:
         logging.warning("No features available for distribution plotting after filtering.")
         return


    logging.info(f"Plotting distributions for {len(selected_features)} selected features from original training set...")
    # Determine grid size for subplots
    n_cols = 2
    n_rows = (len(selected_features) * 2 + n_cols - 1) // n_cols # Each feature gets 2 plots (hist and box)
    if n_rows == 0 and len(selected_features) > 0: n_rows = 1 # Handle case with just one feature selected

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, n_rows * 3.5))
    # Ensure axes is always an array even if n_rows=1, n_cols=1, n_rows=0
    if n_rows == 1 and n_cols == 1:
        axes = np.array([axes])
    elif n_rows == 0: # No subplots needed if no features
        plt.close(fig) # Close the empty figure
        return
    axes = axes.flatten()


    current_plot_idx = 0
    for feat in selected_features:
        # Use data from original training df
        # Handle potential errors if a selected feature column was dropped due to all NaNs after imputation
        if feat in df.columns:
            sns.histplot(data=df, x=feat, hue='label', kde=True, ax=axes[current_plot_idx])
            axes[current_plot_idx].set_title(f'Distribution of {feat}')
            axes[current_plot_idx].set_xlabel("") # Clear x label on subplots
            current_plot_idx += 1

            sns.boxplot(data=df, x='label', y=feat, ax=axes[current_plot_idx])
            axes[current_plot_idx].set_title(f'{feat} by Class')
            axes[current_plot_idx].set_xlabel("") # Clear x label on subplots
            current_plot_idx += 1
        else:
             logging.warning(f"Feature {feat} not found in DataFrame for plotting distribution.")
             # We selected this feature but it wasn't in the DataFrame after imputation.
             # The axes for it will remain unused and hidden later.

    # Hide any unused subplots
    for j in range(current_plot_idx, len(axes)):
        axes[j].set_visible(False)


    plt.tight_layout()
    if CONFIG['experiment']['save_results']:
         plt.savefig(os.path.join(RESULTS_PATH, 'eda_feature_distributions_train_orig.png'))
    plt.show()

    # Correlation matrix (excluding the 'label' column)
    logging.info("Plotting feature correlation matrix for original training set...")
    # Drop columns with all NaNs *before* computing correlation
    df_corr = df.drop('label', axis=1).dropna(axis=1, how='all')
    if not df_corr.empty and df_corr.shape[1] > 1: # Need at least 2 columns for correlation
        corr_matrix = df_corr.corr()
        plt.figure(figsize=(max(8, df_corr.shape[1]*0.6), max(6, df_corr.shape[1]*0.5))) # Adjust size based on num features
        sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', fmt=".2f")
        plt.title('Feature Correlation Matrix (Original Training Set)')
        plt.tight_layout()
        if CONFIG['experiment']['save_results']:
            plt.savefig(os.path.join(RESULTS_PATH, 'eda_correlation_matrix_train_orig.png'))
        plt.show()
    elif df_corr.shape[1] <= 1:
        logging.warning(f"Correlation matrix cannot be computed: only {df_corr.shape[1]} feature column(s) remain after dropping NaNs.")
    else:
        logging.warning("Correlation matrix cannot be computed: DataFrame is empty after dropping NaNs.")

    logging.info("EDA complete.")

# Preprocessing
def prepare_data():
    """
    Extracts features, splits data, scales, and applies SMOTE if configured.
    Returns original training data split for EDA, and processed data for modeling.
    """
    X, y, feature_names = extract_all_features()
    logging.info(f"Extracted data: X shape {X.shape}, y shape {y.shape}")

    if X.shape[0] == 0:
         raise SystemExit("No data extracted after filtering. Cannot proceed with preparation.")

    # --- Split data FIRST ---
    X_tr_orig, X_te_orig, y_tr_orig, y_te = train_test_split(X, y, test_size=CONFIG['preprocessing']['test_size'], stratify=y, random_state=CONFIG['experiment']['random_state'])
    logging.info(f"Data split: Original Training {X_tr_orig.shape[0]} samples, Test {X_te_orig.shape[0]} samples")
    logging.info(f"Original Training class distribution: 0={np.sum(y_tr_orig==0)}, 1={np.sum(y_tr_orig==1)}")
    logging.info(f"Test class distribution: 0={np.sum(y_te==0)}, 1={np.sum(y_te==1)}")


    # --- Impute NaNs ---
    # Using mean imputation as a simple strategy. Consider more sophisticated methods.
    logging.info("Imputing potential NaNs in features...")
    # Calculate mean *only* on the training split to avoid data leakage
    imputer_means = np.nanmean(X_tr_orig, axis=0)
    # Handle case where a column is all NaNs in the training data - replace mean with 0 or another strategy
    imputer_means[np.isnan(imputer_means)] = 0 # Replace NaN means with 0

    X_tr_orig_imputed = np.nan_to_num(X_tr_orig, nan=imputer_means)
    X_te_orig_imputed = np.nan_to_num(X_te_orig, nan=imputer_means)
    logging.info("NaN imputation complete.")


    # --- Scale Data ---
    scaler = StandardScaler().fit(X_tr_orig_imputed) # Fit scaler only on imputed original training data
    X_tr_s = scaler.transform(X_tr_orig_imputed)
    X_te_s = scaler.transform(X_te_orig_imputed)
    logging.info("Data scaled.")

    # --- Apply SMOTE (if configured) ---
    # X_tr_s and y_tr will be the data used for training the model
    if CONFIG['preprocessing']['use_smote']:
        logging.info("Applying SMOTE...")
        smote = SMOTE(random_state=CONFIG['experiment']['random_state'])
        X_tr_s_smote, y_tr_smote = smote.fit_resample(X_tr_s, y_tr_orig) # Apply SMOTE to scaled data
        logging.info(f"After SMOTE: Training {X_tr_s_smote.shape[0]} samples")
        logging.info(f"After SMOTE class distribution: 0={np.sum(y_tr_smote==0)}, 1={np.sum(y_tr_smote==1)}")
        X_tr_for_model, y_tr_for_model = X_tr_s_smote, y_tr_smote
    else:
        # If SMOTE is not used, the training data for the model is just the scaled, imputed original training data
        X_tr_for_model, y_tr_for_model = X_tr_s, y_tr_orig
        logging.info("SMOTE is disabled.")


    # Return data for model training, testing, scaler, feature names, AND original training data (before imputation/scaling) for EDA
    return X_tr_for_model, y_tr_for_model, X_te_s, y_te, scaler, feature_names, X_tr_orig, y_tr_orig

# Build model
def build_stacking_model():
    """Builds the stacking classifier model."""
    ests = []
    mcfg = CONFIG['modeling']['estimators']
    # Use random_state for reproducible base models
    ests.append(('rf', RandomForestClassifier(**mcfg['rf'], random_state=CONFIG['experiment']['random_state'])))
    # Note: SVM can be slow, consider LinearSVC for speed if needed
    ests.append(('svm', SVC(**mcfg['svm'], random_state=CONFIG['experiment']['random_state'])))
    # XGBoost requires numerical labels, ensure y is 0/1
    ests.append(('xgb', xgb.XGBClassifier(**mcfg['xgb'], random_state=CONFIG['experiment']['random_state'])))
    ests.append(('mlp', MLPClassifier(**mcfg['mlp'], random_state=CONFIG['experiment']['random_state'])))

    # Meta-learner also needs random_state if applicable (LogisticRegression does)
    final_estimator = LogisticRegression(**CONFIG['modeling']['final_estimator'], random_state=CONFIG['experiment']['random_state'])

    model = StackingClassifier(
        estimators=ests,
        final_estimator=final_estimator,
        cv=CONFIG['modeling']['cv_folds'], # CV for generating meta-features
        stack_method=CONFIG['modeling']['stack_method'],
        n_jobs=CONFIG['modeling']['n_jobs'],
        passthrough=CONFIG['modeling']['passthrough']
    )
    logging.info("Stacking model built.")
    return model

# Evaluation and Analysis
def plot_confusion_matrix(y_true, y_pred, title='Confusion Matrix', filename="confusion_matrix.png"):
    """Plots and saves the confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[0, 1])
    fig, ax = plt.subplots(figsize=(6, 6))
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    plt.title(title)
    if CONFIG['experiment']['save_results']:
        plt.savefig(os.path.join(RESULTS_PATH, filename))
    plt.show()
    return cm

def plot_roc_pr_curves(model, X_test, y_test, filename_roc="roc_curve.png", filename_pr="pr_curve.png"):
    """Plots and saves ROC and Precision-Recall curves."""
    logging.info("Plotting ROC and PR curves...")
    try:
        # Ensure there are at least two classes in y_test for meaningful curves
        if len(np.unique(y_test)) < 2:
             logging.warning("Skipping ROC/PR curves: only one class present in test set.")
             return None # Cannot plot curves

        y_prob = model.predict_proba(X_test)[:, 1]

        # ROC Curve
        fig_roc, ax_roc = plt.subplots(figsize=(6, 6))
        roc_disp = RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax_roc)
        # Calculate AUC explicitly for logging/saving
        roc_auc = roc_auc_score(y_test, y_prob)
        plt.title(f'ROC Curve (AUC = {roc_auc:.4f})')
        if CONFIG['experiment']['save_results']:
             plt.savefig(os.path.join(RESULTS_PATH, filename_roc))
        plt.show()

        # Precision-Recall Curve
        fig_pr, ax_pr = plt.subplots(figsize=(6, 6))
        pr_disp = PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=ax_pr)
        # Calculate AUC-PR explicitly
        precision, recall, _ = precision_recall_curve(y_test, y_prob) # Need to import precision_recall_curve
        pr_auc = auc(recall, precision)
        plt.title(f'Precision-Recall Curve (AUC-PR = {pr_auc:.4f})')

        if CONFIG['experiment']['save_results']:
             plt.savefig(os.path.join(RESULTS_PATH, filename_pr))
        plt.show()

        return y_prob # Return probabilities for threshold analysis

    except Exception as e:
        logging.error(f"Error plotting ROC/PR curves: {e}")
        import traceback
        logging.error(traceback.format_exc())
        return None


def find_optimal_threshold(y_true, y_prob, metric_to_optimize="f1"):
    """Finds the optimal threshold based on a specified metric."""
    logging.info(f"Finding optimal threshold based on {metric_to_optimize}...")

    # Ensure there are at least two classes in the true labels
    if len(np.unique(y_true)) < 2:
         logging.warning("Cannot find optimal threshold: only one class present in true labels.")
         # Return default threshold and report based on it
         default_threshold = 0.5
         y_pred_default = (y_prob >= default_threshold).astype(int)
         default_report = classification_report(y_true, y_pred_default, output_dict=True, zero_division=0)
         return default_threshold, default_report


    thresholds = np.linspace(0.001, 1, 1000) # Use 1000 thresholds for better precision
    scores = []

    for th in thresholds:
        y_pred_th = (y_prob >= th).astype(int)
        # Calculate the specific score for this threshold
        try:
            if metric_to_optimize == "f1":
                # Use the f1_score function directly, specifying pos_label=1 and zero_division=0
                # Check if class 1 is present in y_true to avoid errors in metric function
                if 1 in y_true:
                     score = f1_score(y_true, y_pred_th, pos_label=1, zero_division=0)
                else: # If class 1 is not in y_true, score is irrelevant/0
                     score = 0.0 # Or np.nan
            elif metric_to_optimize == "recall":
                # Use recall_score function directly
                if 1 in y_true:
                    score = recall_score(y_true, y_pred_th, pos_label=1, zero_division=0)
                else:
                    score = 0.0
            elif metric_to_optimize == "precision":
                 # Use precision_score function directly
                 if 1 in y_true:
                      score = precision_score(y_true, y_pred_th, pos_label=1, zero_division=0)
                 else:
                      score = 0.0 # Precision is trickier with no true positives, 0.0 is a convention here
            elif metric_to_optimize == "accuracy":
                 # Accuracy is overall, doesn't need pos_label
                 score = accuracy_score(y_true, y_pred_th)
            else:
                logging.warning(f"Unknown metric '{metric_to_optimize}' for threshold optimization. Using F1-score (class 1).")
                metric_to_optimize = "f1"
                if 1 in y_true:
                     score = f1_score(y_true, y_pred_th, pos_label=1, zero_division=0)
                else:
                     score = 0.0

            scores.append(score)
        except Exception as e:
            # This catches potential errors in metric calculation for edge thresholds
            logging.debug(f"Error calculating {metric_to_optimize} for threshold {th:.4f}: {e}. Appending NaN.")
            scores.append(np.nan) # Append NaN if calculation fails


    # Handle case where all scores are NaN or 0 (e.g., no positive predictions ever)
    scores = np.array(scores) # Convert list to numpy array for nanargmax
    # Check if there's any valid score greater than 0. If not, argmax will fail or return 0th index which might not be optimal
    if not scores.size or np.all(np.isnan(scores)) or np.all(scores <= 0):
        logging.warning(f"Could not find a meaningful optimal threshold ({metric_to_optimize}). All scores were NaN or <= 0. Returning default 0.5 threshold.")
        optimal_threshold = 0.5
    else:
        # Find the index of the maximum score, ignoring NaNs
        optimal_idx = np.nanargmax(scores)
        optimal_threshold = thresholds[optimal_idx]
        optimal_score = scores[optimal_idx]
        logging.info(f"Optimal threshold ({metric_to_optimize} on class 1): {optimal_threshold:.4f} (Score: {optimal_score:.4f})")


    # Calculate full classification report at the determined optimal threshold
    y_pred_optimal = (y_prob >= optimal_threshold).astype(int)
    optimal_report = classification_report(y_true, y_pred_optimal, output_dict=True, zero_division=0)

    logging.info("Classification Report at Optimal Threshold:")
    print(classification_report(y_true, y_pred_optimal, zero_division=0))


    # Plot confusion matrix at optimal threshold
    if CONFIG['evaluation']['plot_cm']:
         plot_confusion_matrix(y_true, y_pred_optimal, title=f'Confusion Matrix (Optimal Threshold {optimal_threshold:.4f})', filename="confusion_matrix_optimal_threshold.png")

    return optimal_threshold, optimal_report

def plot_feature_importances(model, feature_names, filename="feature_importances.png"):
    """
    Plots feature importances from base estimators that support it.
    Note: This shows importance *for the base learners* on original features,
    not the meta-learner's importance on base learner outputs.
    """
    logging.info("Plotting feature importances from base estimators...")
    importances = []
    sources = []
    # zip original estimator tuples with fitted estimators
    for (name, _), fitted in zip(model.estimators, model.estimators_):
        if hasattr(fitted, 'feature_importances_'):
            importances.append(fitted.feature_importances_)
            sources.append(name)
        elif hasattr(fitted, 'coef_'):
            # For linear models, use absolute coefficients as a proxy for importance
            # Ensure coef_ is 1D or 2D with shape (1, n_features)
            coef = fitted.coef_
            if coef.ndim > 2 or (coef.ndim == 2 and coef.shape[0] > 1):
                 logging.debug(f"Estimator {name} has coef_ with unexpected shape {coef.shape}. Skipping importance.")
                 continue

            coef = coef.ravel() # Flatten if it's shape (1, n)
            # Ensure the number of coefficients matches the expected feature size
            if len(coef) == len(feature_names):
                 importances.append(np.abs(coef))
                 sources.append(name)
            else:
                 logging.debug(f"Estimator {name} coef_ length ({len(coef)}) does not match feature names length ({len(feature_names)}). Skipping importance.")
        else:
            logging.debug(f"Estimator {name} does not have feature_importances_ or coef_.") # Changed to debug


    if not importances:
        logging.warning("No feature importances available from base estimators that support it.")
        return

    # Ensure all importance arrays have the same size as feature_names
    expected_size = len(feature_names)
    # Filter out importance arrays that don't match the expected size
    valid_importances = [imp for imp in importances if len(imp) == expected_size]

    if not valid_importances:
        logging.warning("Feature importance arrays have inconsistent sizes after filtering. Cannot plot.")
        return

    # Calculate mean importance across valid estimators
    # Handle potential NaNs within importance arrays
    avg_imp = np.nanmean(valid_importances, axis=0) # Use nanmean
    # Replace potential NaNs in avg_imp if a feature was NaN for all estimators that provided importance
    avg_imp = np.nan_to_num(avg_imp, nan=0)


    # Create DataFrame for plotting
    imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': avg_imp})
    # Sort and get top N, ensure we don't ask for more than available features
    top_n = min(CONFIG['evaluation']['feat_imp_top_n'], len(imp_df))
    imp_df = imp_df.sort_values('Importance', ascending=False).head(top_n)

    if imp_df.empty:
        logging.warning("No features left to plot after filtering.")
        return


    plt.figure(figsize=(8, max(6, len(imp_df) * 0.4))) # Adjust figure size based on number of features
    # Add legend=False to silence the FutureWarning
    sns.barplot(y='Feature', hue="Feature"x='Importance', data=imp_df, palette='viridis', legend=False)
    plt.title(f'Top {len(imp_df)} Feature Importances (Avg across Base Estimators)')
    plt.tight_layout()
    if CONFIG['experiment']['save_results']:
        plt.savefig(os.path.join(RESULTS_PATH, filename))
    plt.show()

def analyze_meta_learner(model):
    """Analyzes the meta-learner's coefficients on base estimator outputs."""
    logging.info("Analyzing Meta-Learner...")
    final_estimator = model.final_estimator_
    # Check if the meta-learner is a linear model with coefficients
    if hasattr(final_estimator, 'coef_'):
        coef = final_estimator.coef_
        # Ensure coef_ is 1D or 2D with shape (1, n_features)
        if coef.ndim > 2 or (coef.ndim == 2 and coef.shape[0] > 1):
             logging.warning(f"Meta-learner has coef_ with unexpected shape {coef.shape}. Cannot display importance.")
             return

        coef = coef.ravel() # Flatten if it's shape (1, n)

        # The coefficients correspond to the predictions of the base estimators + original features (if passthrough)
        base_estimator_names = [name for name, _ in model.estimators]

        # If passthrough is False (default), coef_ should match the number of base estimators
        if not model.passthrough:
            if len(coef) == len(base_estimator_names):
                meta_imp = pd.DataFrame({'Base Estimator': base_estimator_names, 'Coefficient (Abs)': np.abs(coef), 'Coefficient': coef})
                logging.info("Meta-Learner Coefficients on Base Estimator Outputs:")
                logging.info(meta_imp.sort_values('Coefficient (Abs)', ascending=False).to_string()) # Use to_string to print full DataFrame
            else:
                logging.warning(f"Meta-learner coefficients length ({len(coef)}) does not match the number of base estimators ({len(base_estimator_names)}). Cannot display meta-learner importance.")
        else:
             # If passthrough is True, coefficients include original features.
             # The order is base estimator outputs first, then original features.
             expected_coef_len = len(base_estimator_names) + len(FEATURE_NAMES)
             if len(coef) == expected_coef_len:
                  logging.info("Meta-Learner Coefficients (Base Estimator Outputs + Original Features):")
                  # Display coefficients for base estimators
                  meta_base_coef = pd.DataFrame({'Base Estimator': base_estimator_names, 'Coefficient (Abs)': np.abs(coef[:len(base_estimator_names)]), 'Coefficient': coef[:len(base_estimator_names)]})
                  logging.info("  Base Estimator Coefficients:")
                  logging.info(meta_base_coef.sort_values('Coefficient (Abs)', ascending=False).to_string())

                  # Display coefficients for original features
                  meta_feat_coef = pd.DataFrame({'Feature': FEATURE_NAMES, 'Coefficient (Abs)': np.abs(coef[len(base_estimator_names):]), 'Coefficient': coef[len(base_estimator_names):]})
                  logging.info("  Original Feature Coefficients:")
                  logging.info(meta_feat_coef.sort_values('Coefficient (Abs)', ascending=False).to_string())

             else:
                  logging.warning(f"Meta-learner coefficients length ({len(coef)}) does not match the expected length ({expected_coef_len}) with passthrough=True.")
                  logging.warning("Cannot confidently display meta-learner importance with passthrough=True.")


    elif hasattr(final_estimator, 'feature_importances_'):
        # If the meta-learner is tree-based (e.g., RandomForestRegressor) and passthrough=False, this shouldn't happen for StackingClassifier meta_estimator
        # If passthrough=True and meta-learner is tree-based, importance includes original features.
         logging.info("Meta-Learner Feature Importances (includes base estimator outputs and potentially original features if passthrough=True):")
         importances = final_estimator.feature_importances_
         logging.info(f"Importances array shape: {importances.shape}")
         if model.passthrough and 'FEATURE_NAMES' in globals():
              base_estimator_names = [name for name, _ in model.estimators]
              expected_imp_len = len(base_estimator_names) + len(FEATURE_NAMES)
              if len(importances) == expected_imp_len:
                   # Display importances for base estimators
                   meta_base_imp = pd.DataFrame({'Base Estimator': base_estimator_names, 'Importance': importances[:len(base_estimator_names)]})
                   logging.info("  Base Estimator Importances:")
                   logging.info(meta_base_imp.sort_values('Importance', ascending=False).to_string())

                   # Display importances for original features
                   meta_feat_imp = pd.DataFrame({'Feature': FEATURE_NAMES, 'Importance': importances[len(base_estimator_names):]})
                   logging.info("  Original Feature Importances:")
                   logging.info(meta_feat_imp.sort_values('Importance', ascending=False).to_string())
              else:
                   logging.warning(f"Meta-learner importance length ({len(importances)}) does not match expected length ({expected_imp_len}) with passthrough=True.")
                   logging.warning("Cannot confidently display meta-learner importance with passthrough=True.")

         else:
              logging.info(f"Importances: {importances}") # Fallback print
    else:
        logging.warning("Meta-learner does not have a 'coef_' or 'feature_importances_' attribute, or its shape is unexpected.")


def save_results(metrics_dict, classification_report_dict, config_dict_original, model, scaler, feature_names):
    """Saves all relevant results to the results directory."""
    logging.info(f"Saving results to {RESULTS_PATH}...")

    try:
        # Create a copy of the config and remove non-serializable parts
        config_to_save = copy.deepcopy(config_dict_original)
        if 'evaluation' in config_to_save and 'metrics' in config_to_save['evaluation']:
            # Remove the list of function objects, keep only the names
             del config_to_save['evaluation']['metrics']
        # Remove the mapping from names to functions
        if 'METRIC_FUNCTIONS' in globals():
             # You might not need to save this mapping if it's hardcoded in the script
             # But if you put it in config, remove it here. It's global now, so not in config_to_save.
             pass


        # Save configuration
        config_path = os.path.join(RESULTS_PATH, "config.json")
        with open(config_path, 'w') as f:
             json.dump(config_to_save, f, indent=4)
        logging.info(f"Configuration saved to {config_path}")

        # Save metrics
        metrics_path = os.path.join(RESULTS_PATH, "evaluation_metrics.json")
        with open(metrics_path, 'w') as f:
            json.dump(metrics_dict, f, indent=4)
        logging.info(f"Metrics saved to {metrics_path}")

        # Save classification report
        report_path = os.path.join(RESULTS_PATH, "classification_report.json")
        with open(report_path, 'w') as f:
            # Ensure the report dictionary is JSON serializable (might contain numpy types)
            # Simple conversion to standard Python types might be needed if errors persist
            def convert_numpy_types(obj):
                 if isinstance(obj, np.generic):
                      return obj.item() # Convert numpy scalar to Python scalar
                 if isinstance(obj, dict):
                      return {k: convert_numpy_types(v) for k, v in obj.items()}
                 if isinstance(obj, list):
                      return [convert_numpy_types(item) for item in obj]
                 # Convert numpy arrays to lists
                 if isinstance(obj, np.ndarray):
                     return obj.tolist()
                 return obj

            serializable_report = convert_numpy_types(classification_report_dict)
            json.dump(serializable_report, f, indent=4)
        logging.info(f"Classification report saved to {report_path}")


        # Save trained model
        model_path = os.path.join(RESULTS_PATH, "stacked_model.joblib")
        joblib.dump(model, model_path)
        logging.info(f"Trained model saved to {model_path}")

        # Save scaler
        scaler_path = os.path.join(RESULTS_PATH, "scaler.joblib")
        joblib.dump(scaler, scaler_path)
        logging.info(f"Scaler saved to {scaler_path}")

        # Save feature names (useful when loading model later)
        feat_names_path = os.path.join(RESULTS_PATH, "feature_names.json")
        with open(feat_names_path, 'w') as f:
             json.dump(feature_names, f, indent=4)
        logging.info(f"Feature names saved to {feat_names_path}")

        logging.info("All results saved.")

    except Exception as e:
        logging.error(f"Error during saving results: {e}")
        import traceback
        logging.error(traceback.format_exc())


# Main execution
def main():
    """Main function to run the prediction pipeline."""
    logging.info("Starting SCD Prediction Pipeline...")

    # --- 1. Data Preparation ---
    try:
        # prepare_data now returns original train data split for EDA
        X_tr_for_model, y_tr_for_model, X_te_s, y_te, scaler, feature_names, X_tr_orig, y_tr_orig = prepare_data()
    except SystemExit as e:
         logging.error(f"Data preparation failed: {e}")
         return # Exit if no data is available
    except Exception as e:
        logging.error(f"An unexpected error occurred during data preparation: {e}")
        import traceback
        logging.error(traceback.format_exc())
        return

    # --- 2. Exploratory Data Analysis (Optional) ---
    # Run EDA on the *original* training split (before scaling and SMOTE)
    if CONFIG['experiment']['run_eda']:
        # Ensure X_tr_orig and y_tr_orig are not empty before running EDA
        if X_tr_orig.shape[0] > 0:
            run_eda(X_tr_orig, y_tr_orig, feature_names)
        else:
             logging.warning("Skipping EDA: Original training data is empty.")


    # --- 3. Model Building and Training ---
    model = build_stacking_model()
    logging.info("Training stacking model...")
    try:
        # Ensure training data is not empty
        if X_tr_for_model.shape[0] > 0:
            model.fit(X_tr_for_model, y_tr_for_model)
            logging.info("Model training complete.")
            model_fitted = True
        else:
             logging.error("Training data is empty. Skipping model training.")
             model_fitted = False
             # Set model to None or handle gracefully if subsequent steps use it
             model = None

    except Exception as e:
        logging.error(f"An error occurred during model training: {e}")
        import traceback
        logging.error(traceback.format_exc())
        model_fitted = False
        model = None # Ensure model is None if fit failed


    # --- 4. Model Evaluation (on Test Set) ---
    # Only evaluate if the model was successfully fitted and test data exists
    if model_fitted and X_te_s.shape[0] > 0:
        logging.info("Evaluating model on test set...")
        metrics_results = {}
        classification_rep = {}
        try:
            y_pred = model.predict(X_te_s)
            # Ensure there are at least two classes in y_te to calculate probabilities meaningfully
            # and for AUC, PR AUC, etc.
            if len(np.unique(y_te)) > 1:
                 y_prob = model.predict_proba(X_te_s)[:, 1] # Probability of the positive class (SCD)
                 logging.debug("Probabilities calculated.")
            else:
                 y_prob = None
                 logging.warning("Only one class present in test set. Cannot calculate probabilities, AUC, or perform threshold optimization.")


            # Calculate standard metrics (using default threshold 0.5)
            logging.info("Classification Report (Default Threshold 0.5):")
            # Ensure the report is generated even if only one class is present
            classification_rep = classification_report(y_te, y_pred, output_dict=True, zero_division=0)
            print(classification_report(y_te, y_pred, zero_division=0)) # Also print to console

            for metric_name in CONFIG['evaluation']['metrics_to_report']:
                 try:
                     # Use the METRIC_FUNCTIONS mapping
                     metric_func = METRIC_FUNCTIONS.get(metric_name)
                     if metric_func is None:
                          logging.warning(f"Metric function not found for name: {metric_name}")
                          metrics_results[metric_name] = None
                          continue

                     if metric_name == "AUC":
                         # AUC uses probabilities and requires >1 class
                         if y_prob is not None and len(np.unique(y_te)) > 1:
                             score = metric_func(y_te, y_prob)
                             logging.info(f"{metric_name}: {score:.4f}")
                             metrics_results[metric_name] = score
                         else:
                              metrics_results[metric_name] = None # Indicate not computed

                     elif metric_name in ["F1-Score", "Recall", "Precision"]:
                         # These metrics typically focus on the positive class in imbalanced scenarios
                         # Use .get() safely to retrieve the score from the report dictionary
                         # Classification report uses lowercase keys with underscores (e.g., 'f1-score' in dict)
                         # Special case for f1-score in dictionary lookup
                         report_key = metric_name.lower().replace('-', '_')
                         if report_key == 'f1_score': report_key = 'f1-score' # The dict uses 'f1-score' key

                         score = classification_rep.get('1', {}).get(report_key)

                         if score is not None:
                              logging.info(f"{metric_name} (Class 1): {score:.4f}")
                              metrics_results[metric_name] = score
                         else:
                              # This case should be rare with zero_division=0, but added safety
                              logging.debug(f"Could not retrieve {metric_name} for Class 1 from classification report dictionary.")
                              # Attempt to compute directly using the function if report failed? No, stick to report as primary source for consistency.
                              metrics_results[metric_name] = None


                     elif metric_name == "Accuracy": # Accuracy
                         score = metric_func(y_te, y_pred) # Accuracy uses predicted class
                         logging.info(f"{metric_name}: {score:.4f}")
                         metrics_results[metric_name] = score

                     # Add other metrics here if needed...

                 except Exception as e:
                     logging.warning(f"Could not compute {metric_name}: {e}")
                     metrics_results[metric_name] = None


            # Confusion Matrix (Default Threshold 0.5)
            if CONFIG['evaluation']['plot_cm']:
                plot_confusion_matrix(y_te, y_pred, title='Confusion Matrix (Default Threshold 0.5)')

            # ROC and Precision-Recall Curves
            if CONFIG['evaluation']['plot_roc_pr'] and y_prob is not None:
                 # plot_roc_pr_curves handles the case where only one class is present internally
                 plot_roc_pr_curves(model, X_te_s, y_te)
            elif CONFIG['evaluation']['plot_roc_pr']:
                 logging.warning("Skipping ROC/PR curve plotting: Probabilities not available (likely due to only one class in test set).")


            # Threshold Optimization
            # Only perform threshold optimization if probabilities are available and >1 class in y_te
            if y_prob is not None and len(np.unique(y_te)) > 1:
                 optimal_threshold, optimal_report_dict = find_optimal_threshold(y_te, y_prob, CONFIG['evaluation']['optimal_threshold_metric'])
                 metrics_results[f"Optimal_Threshold ({CONFIG['evaluation']['optimal_threshold_metric']})"] = optimal_threshold
                 # Store key metrics at optimal threshold from the optimal report dict
                 metrics_results["Optimal_Threshold_Accuracy"] = optimal_report_dict.get('accuracy')
                 metrics_results["Optimal_Threshold_Recall_SCD"] = optimal_report_dict.get('1', {}).get('recall') # Safely access nested dict
                 metrics_results["Optimal_Threshold_Precision_SCD"] = optimal_report_dict.get('1', {}).get('precision')
                 metrics_results["Optimal_Threshold_F1_SCD"] = optimal_report_dict.get('1', {}).get('f1-score')
                 metrics_results["Optimal_Threshold_Specificity"] = optimal_report_dict.get('0', {}).get('recall') # Recall for class 0 is specificity

                 # Store the optimal threshold report within the classification report results
                 classification_rep["Optimal_Threshold_Report"] = optimal_report_dict
            else:
                 logging.warning("Skipping threshold optimization: Probabilities not available or only one class present in test set.")


        except Exception as e:
            logging.error(f"An error occurred during model evaluation: {e}")
            import traceback
            logging.error(traceback.format_exc())
            # Keep existing metrics_results and classification_rep which might be partially filled

    elif X_te_s.shape[0] == 0:
         logging.warning("Test data is empty. Skipping model evaluation.")
         metrics_results = {}
         classification_rep = {}
    else: # model_fitted is False
         logging.warning("Model was not fitted successfully. Skipping model evaluation.")
         metrics_results = {}
         classification_rep = {}


    # --- 5. Model Analysis ---
    # Only analyze model if it was fitted
    if model_fitted:
        # Feature Importance (from base estimators)
        # Check if feature_names is populated before plotting importance
        if CONFIG['evaluation']['plot_feat_imp'] and feature_names and len(feature_names) > 0:
             plot_feature_importances(model, feature_names, filename="base_feature_importances.png")
        elif CONFIG['evaluation']['plot_feat_imp']:
             logging.warning("Skipping feature importance plot: feature_names is empty or not set.")

        # Meta-learner analysis
        analyze_meta_learner(model)
    else:
         logging.warning("Skipping model analysis: Model was not fitted successfully.")


    # --- 6. Saving Results (Optional) ---
    # Save results even if evaluation/analysis had issues, as long as save_results is true
    # Pass the original CONFIG dictionary
    if CONFIG['experiment']['save_results']:
        save_results(metrics_results, classification_rep, CONFIG, model, scaler, feature_names)


    logging.info("SCD Prediction Pipeline finished.")

if __name__=='__main__':
    main()